<a href="https://colab.research.google.com/github/gabrielladiogenes/LOC-VIEW-AI/blob/main/loc_view_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# LOC VIEW AI
# Intelligent Location Visualization

import gradio as gr

print("LOC VIEW AI initialized!")
print("Environment successfully prepared.")

LOC VIEW AI initialized!
Environment successfully prepared.


In [ ]:

# ============================================================
# LOC VIEW AI v5.9.3
# Intelligent Location Visualization
# Google Colab — Single Cell
# ============================================================

import sys
import subprocess
import importlib.util

for package in ("gradio", "requests"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import re
import time
import unicodedata
import threading

from difflib import SequenceMatcher
from urllib.parse import urlencode, quote
from html import escape

import requests
import gradio as gr


# ============================================================
# CONFIGURATION
# ============================================================

VERSION = "5.9.3"

API = "https://nominatim.openstreetmap.org/search"

USER_AGENT = (
    "LOCViewAI/5.9.3 "
    "(educational location visualization application)"
)

SESSION = requests.Session()
LOCK = threading.Lock()
LAST_REQUEST = [0.0]

STATES = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AP": "Amapá",
    "AM": "Amazonas",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MT": "Mato Grosso",
    "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais",
    "PA": "Pará",
    "PB": "Paraíba",
    "PR": "Paraná",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul",
    "RO": "Rondônia",
    "RR": "Roraima",
    "SC": "Santa Catarina",
    "SP": "São Paulo",
    "SE": "Sergipe",
    "TO": "Tocantins"
}

NUMWORDS = {
    "zero": 0,
    "um": 1,
    "uma": 1,
    "dois": 2,
    "duas": 2,
    "tres": 3,
    "quatro": 4,
    "cinco": 5,
    "seis": 6,
    "sete": 7,
    "oito": 8,
    "nove": 9,
    "dez": 10,
    "onze": 11,
    "doze": 12,
    "treze": 13,
    "quatorze": 14,
    "quinze": 15,
    "dezesseis": 16,
    "dezessete": 17,
    "dezoito": 18,
    "dezenove": 19,
    "vinte": 20,
    "trinta": 30,
    "quarenta": 40,
    "cinquenta": 50,
    "sessenta": 60,
    "setenta": 70,
    "oitenta": 80,
    "noventa": 90
}


# ============================================================
# TEXT NORMALIZATION
# ============================================================

def norm(value):

    value = unicodedata.normalize(
        "NFKD",
        str(value or "")
    )

    value = "".join(
        char
        for char in value
        if not unicodedata.combining(char)
    )

    return re.sub(
        r"\s+",
        " ",
        re.sub(
            r"[^a-z0-9]+",
            " ",
            value.lower()
        )
    ).strip()


def normalize_street(value):

    value = norm(value)

    value = re.sub(
        r"^r\b",
        "rua",
        value
    )

    value = re.sub(
        r"^av\b",
        "avenida",
        value
    )

    words = value.split()

    result = []
    i = 0

    while i < len(words):

        if words[i] in NUMWORDS:

            number = NUMWORDS[words[i]]
            j = i + 1

            if (
                number >= 20
                and number % 10 == 0
                and j + 1 < len(words)
                and words[j] == "e"
                and words[j + 1] in NUMWORDS
                and NUMWORDS[words[j + 1]] < 10
            ):

                number += NUMWORDS[
                    words[j + 1]
                ]

                j += 2

            result.append(str(number))
            i = j

        else:

            result.append(words[i])
            i += 1

    return " ".join(result)


def street_equal(a, b):

    x = normalize_street(a)
    y = normalize_street(b)

    if not x or not y:
        return False

    if x == y:
        return True

    return (
        len(x) > 10
        and len(y) > 10
        and SequenceMatcher(
            None,
            x,
            y
        ).ratio() >= 0.94
    )


def canonical_state(value):

    key = norm(value)

    for abbreviation, full_name in STATES.items():

        if key in (
            norm(abbreviation),
            norm(full_name)
        ):
            return norm(full_name)

    return key


# ============================================================
# ADDRESS PARSER
# ============================================================

def parse_address(raw):

    raw = re.sub(
        r"\s+",
        " ",
        (raw or "").strip()
    )

    if not raw:

        raise ValueError(
            "Enter a complete address."
        )

    postal_matches = re.findall(
        r"(?<!\d)\d{5}-?\d{3}(?!\d)",
        raw
    )

    postal_matches = [
        re.sub(r"\D", "", item)
        for item in postal_matches
    ]

    if len(set(postal_matches)) > 1:

        raise ValueError(
            "Conflicting postal codes were "
            "found in the address."
        )

    postal_code = (
        postal_matches[0]
        if postal_matches
        else ""
    )

    clean = re.sub(
        r"(?<!\d)\d{5}-?\d{3}(?!\d)",
        "",
        raw
    )

    clean = re.sub(
        r"\s*/\s*",
        " / ",
        clean
    )

    clean = re.sub(
        r"\s*,\s*",
        ", ",
        clean
    )

    clean = re.sub(
        r"\s*,\s*,+",
        ",",
        clean
    ).strip(" ,-")

    parts = [
        part.strip(" ,")
        for part in clean.split(",")
        if part.strip(" ,")
    ]

    if not parts:

        raise ValueError(
            "Enter a street and an exact house number."
        )

    street = parts[0]
    number = ""
    tail = parts[1:]

    # Identify house number without treating
    # numbers in street names as house numbers.

    match = re.search(
        r"(?:,\s*|\s+(?:n[º°.]?|numero|número)\s*)"
        r"(\d+[A-Za-z]?)\b",
        clean,
        re.I
    )

    if (
        match
        and match.start() < len(parts[0]) + 5
    ):

        number = match.group(1)

        if re.search(
            r"\d+[A-Za-z]?\s*$",
            parts[0]
        ):

            street = re.sub(
                r"\s+(?:n[º°.]?|numero|número)?"
                r"\s*\d+[A-Za-z]?\s*$",
                "",
                parts[0],
                flags=re.I
            )

        elif (
            tail
            and re.fullmatch(
                r"\d+[A-Za-z]?(?:\s*-.*)?",
                tail[0]
            )
        ):

            tail.pop(0)

    if not number and tail:

        number_match = re.match(
            r"^(\d+[A-Za-z]?)"
            r"(?:\s*-\s*(.*))?$",
            tail[0]
        )

        if number_match:

            number = number_match.group(1)

            remainder = (
                number_match.group(2) or ""
            ).strip()

            tail = (
                [remainder]
                if remainder
                else []
            ) + tail[1:]

    if not number:

        number_match = re.search(
            r"\s+(\d+[A-Za-z]?)$",
            street
        )

        if number_match:

            number = number_match.group(1)

            street = street[
                :number_match.start()
            ].strip()

    if not number:

        raise ValueError(
            "Enter the exact house number. "
            "The address does not include one."
        )

    if not re.match(
        r"(?i)^(?:rua\b|r\.(?=\s)|"
        r"avenida\b|av\.(?=\s)|"
        r"travessa\b|alameda\b|"
        r"praca\b|praça\b|"
        r"estrada\b|rodovia\b|"
        r"largo\b|viela\b)",
        street
    ):

        raise ValueError(
            "Enter the street type and name "
            "(e.g., Rua or Avenida)."
        )

    extra = ", ".join(tail)

    extra = re.sub(
        r"\s*/\s*",
        ", ",
        extra
    )

    extra = re.sub(
        r"\s+-\s+(?=[A-Za-zÀ-ÿ ]+\s*(?:,|$))",
        ", ",
        extra
    )

    locations = [
        item.strip()
        for item in extra.split(",")
        if item.strip()
    ]

    locations = [
        item
        for item in locations
        if norm(item) not in (
            "brasil",
            "brazil"
        )
    ]

    state = ""
    city = ""
    neighborhood = ""

    valid_states = {
        norm(value)
        for value in STATES.values()
    }

    state_index = next(
        (
            index
            for index in range(
                len(locations) - 1,
                -1,
                -1
            )
            if canonical_state(
                locations[index]
            ) in valid_states
        ),
        None
    )

    if state_index is not None:

        state = locations[state_index]

        if state_index >= 1:
            city = locations[state_index - 1]

        if state_index >= 2:
            neighborhood = locations[
                state_index - 2
            ]

    elif locations:

        city = locations[-1]

        if len(locations) > 1:
            neighborhood = locations[-2]

    if not city:

        raise ValueError(
            "Enter the city and state "
            "(e.g., Fortaleza, CE)."
        )

    if not state:

        raise ValueError(
            "Enter the state (e.g., CE or SP)."
        )

    return {
        "street": street.strip(),
        "number": number.upper(),
        "city": city,
        "state": state,
        "neighborhood": neighborhood,
        "postal_code": postal_code
    }


# ============================================================
# POSTAL CODE VALIDATION
# ============================================================

def postal_input(value):

    value = (value or "").strip()

    if not value:
        return ""

    if not re.fullmatch(
        r"\d{5}-?\d{3}",
        value
    ):

        raise ValueError(
            "Invalid postal code. "
            "Use 8 digits, e.g., 60840-000."
        )

    return re.sub(
        r"\D",
        "",
        value
    )


# ============================================================
# NOMINATIM SEARCH
# ============================================================

def search_api(params):

    with LOCK:

        elapsed = (
            time.monotonic()
            - LAST_REQUEST[0]
        )

        wait = 1.15 - elapsed

        if wait > 0:
            time.sleep(wait)

        LAST_REQUEST[0] = time.monotonic()

        response = SESSION.get(
            API,
            params={
                "format": "jsonv2",
                "addressdetails": 1,
                "limit": 20,
                "countrycodes": "br",
                **params
            },
            headers={
                "User-Agent": USER_AGENT
            },
            timeout=25
        )

        response.raise_for_status()

    return response.json()


def fields(result):

    address = result.get(
        "address"
    ) or {}

    street = next(
        (
            address[key]
            for key in (
                "road",
                "pedestrian",
                "residential",
                "footway",
                "street",
                "square"
            )
            if address.get(key)
        ),
        ""
    )

    city = next(
        (
            address[key]
            for key in (
                "city",
                "town",
                "municipality",
                "village"
            )
            if address.get(key)
        ),
        ""
    )

    return {
        "street": street,

        "number": str(
            address.get(
                "house_number",
                ""
            )
        ).upper().strip(),

        "city": city,

        "state": address.get(
            "state",
            ""
        ),

        "country": address.get(
            "country_code",
            ""
        ),

        "postal": re.sub(
            r"\D",
            "",
            address.get(
                "postcode",
                ""
            )
        )
    }


def check(result, parsed, exact=False):

    address = fields(result)

    if address["country"] != "br":
        return False

    if norm(
        address["city"]
    ) != norm(
        parsed["city"]
    ):
        return False

    if canonical_state(
        address["state"]
    ) != canonical_state(
        parsed["state"]
    ):
        return False

    if not street_equal(
        address["street"],
        parsed["street"]
    ):
        return False

    if exact:

        if norm(
            address["number"]
        ) != norm(
            parsed["number"]
        ):
            return False

        if (
            parsed["postal_code"]
            and address["postal"]
            and parsed["postal_code"]
            != address["postal"]
        ):
            return False

    return True


# ============================================================
# GOOGLE MAPS LINKS
# ============================================================

def links(parsed, result):

    latitude = result["lat"]
    longitude = result["lon"]

    address_text = (
        f"{parsed['street']}, "
        f"{parsed['number']}, "
        f"{parsed['city']}, "
        f"{parsed['state']}, Brasil"
    )

    maps_url = (
        "https://www.google.com/maps/search/"
        "?api=1&query="
        + quote(address_text)
    )

    street_view_url = (
        "https://www.google.com/maps/@?"
        + urlencode({
            "api": "1",
            "map_action": "pano",
            "viewpoint": (
                f"{latitude},{longitude}"
            )
        })
    )

    return maps_url, street_view_url


# ============================================================
# SVG ICONS
# ============================================================

CHECK_ICON = """
<svg viewBox="0 0 24 24" fill="none"
stroke="currentColor" stroke-width="2"
stroke-linecap="round" stroke-linejoin="round">
<circle cx="12" cy="12" r="10"/>
<path d="m8 12 2.5 2.5L16 9"/>
</svg>
"""

WARNING_ICON = """
<svg viewBox="0 0 24 24" fill="none"
stroke="currentColor" stroke-width="2"
stroke-linecap="round" stroke-linejoin="round">
<circle cx="12" cy="12" r="10"/>
<path d="M12 7v6m0 4h.01"/>
</svg>
"""

MAP_ICON = """
<svg viewBox="0 0 24 24" fill="none"
stroke="currentColor" stroke-width="2"
stroke-linecap="round" stroke-linejoin="round">
<polygon points="3 6 9 3 15 6 21 3 21 18 15 21 9 18 3 21 3 6"/>
<line x1="9" y1="3" x2="9" y2="18"/>
<line x1="15" y1="6" x2="15" y2="21"/>
</svg>
"""

PHOTO_ICON = """
<svg viewBox="0 0 24 24" fill="none"
stroke="currentColor" stroke-width="2"
stroke-linecap="round" stroke-linejoin="round">
<rect x="3" y="3" width="18" height="18" rx="2"/>
<circle cx="8.5" cy="8.5" r="1.5"/>
<path d="m21 15-5-5L5 21"/>
</svg>
"""

ARROW_ICON = """
<svg viewBox="0 0 24 24" fill="none"
stroke="currentColor" stroke-width="2"
stroke-linecap="round" stroke-linejoin="round">
<path d="M7 17 17 7M7 7h10v10"/>
</svg>
"""

PIN_ICON = """
<svg viewBox="0 0 24 24" fill="none"
stroke="currentColor" stroke-width="2"
stroke-linecap="round" stroke-linejoin="round"
width="23" height="23">
<path d="M20 10c0 5-8 12-8 12S4 15 4 10a8 8 0 1 1 16 0Z"/>
<circle cx="12" cy="10" r="2.5"/>
</svg>
"""


# ============================================================
# RESULT COMPONENTS
# ============================================================

def panel(title, details="", kind="warn"):

    icon = (
        CHECK_ICON
        if kind == "ok"
        else WARNING_ICON
    )

    return (
        '<div class="result">'
        f'<div class="status {kind}">'
        f'{icon}'
        f'<span>{escape(title)}</span>'
        '</div>'
        f'<div class="detail">'
        f'{escape(details)}'
        '</div>'
        '</div>'
    )


def success_panel(result, parsed):

    maps_url, street_view_url = links(
        parsed,
        result
    )

    label = escape(
        result.get(
            "display_name",
            ""
        )
    )

    latitude = escape(
        str(result["lat"])
    )

    longitude = escape(
        str(result["lon"])
    )

    return (
        '<div class="result">'

        '<div class="status ok">'
        f'{CHECK_ICON}'
        '<span>Location identified</span>'
        '</div>'

        f'<div class="address">{label}</div>'

        '<div class="coords">'

        '<div class="coord">'
        '<div class="k">LATITUDE</div>'
        f'<div class="v">{latitude}</div>'
        '</div>'

        '<div class="coord">'
        '<div class="k">LONGITUDE</div>'
        f'<div class="v">{longitude}</div>'
        '</div>'

        '</div>'

        f'<a class="action primary" '
        f'href="{escape(maps_url, quote=True)}" '
        'target="_blank" rel="noopener noreferrer">'
        f'{MAP_ICON}'
        '<span>VIEW LOCATION ON MAPS</span>'
        f'{ARROW_ICON}'
        '</a>'

        f'<a class="action secondary" '
        f'href="{escape(street_view_url, quote=True)}" '
        'target="_blank" rel="noopener noreferrer">'
        f'{PHOTO_ICON}'
        '<span>EXPLORE STREET VIEW</span>'
        f'{ARROW_ICON}'
        '</a>'

        '</div>'
    )


# ============================================================
# MAIN SEARCH
# ============================================================

def search_location_ui(raw, postal):

    try:

        parsed = parse_address(raw)
        postal_code = postal_input(postal)

        if (
            postal_code
            and parsed["postal_code"]
            and postal_code != parsed["postal_code"]
        ):

            return panel(
                "Conflicting postal codes",
                "The address and Postal code field "
                "contain different values."
            )

        if postal_code:
            parsed["postal_code"] = postal_code

    except ValueError as error:

        return panel(
            str(error)
        )

    state = STATES.get(
        parsed["state"].upper(),
        parsed["state"]
    )

    base = (
        f"{parsed['street']}, "
        f"{parsed['number']}, "
        f"{parsed['city']}, "
        f"{state}, Brasil"
    )

    queries = [
        {
            "q": base
        },

        {
            "q": (
                f"{parsed['number']} "
                f"{parsed['street']}, "
                f"{parsed['city']}, "
                f"{state}, Brasil"
            )
        },

        {
            "street": (
                f"{parsed['number']} "
                f"{parsed['street']}"
            ),
            "city": parsed["city"],
            "state": state,
            "country": "Brasil"
        }
    ]

    if parsed["postal_code"]:

        queries.insert(
            0,
            {
                "q": (
                    f"{parsed['street']}, "
                    f"{parsed['number']}, "
                    f"{parsed['city']}, "
                    f"{state}, "
                    f"{parsed['postal_code']}, "
                    "Brasil"
                )
            }
        )

    exact_results = []
    street_results = []

    try:

        for query in queries:

            rows = search_api(query)

            exact_results.extend(rows)

            matches = [
                row
                for row in rows
                if check(
                    row,
                    parsed,
                    exact=True
                )
            ]

            if matches:

                return success_panel(
                    matches[0],
                    parsed
                )

        street_queries = [
            {
                "q": (
                    f"{parsed['street']}, "
                    f"{parsed['city']}, "
                    f"{state}, Brasil"
                )
            },

            {
                "street": parsed["street"],
                "city": parsed["city"],
                "state": state,
                "country": "Brasil"
            }
        ]

        for query in street_queries:

            rows = search_api(query)
            street_results.extend(rows)

    except requests.RequestException:

        return panel(
            "Search temporarily unavailable",
            "Please try again in a moment.",
            "bad"
        )

    matches = [
        row
        for row in (
            exact_results
            + street_results
        )
        if check(
            row,
            parsed,
            exact=False
        )
    ]

    if matches:

        return panel(
            "Street located — house number not confirmed",

            (
                f"{matches[0].get('display_name', '')}\n"
                f"Requested house number: "
                f"{parsed['number']}.\n"
                "The exact building location "
                "could not be verified."
            )
        )

    return panel(
        "Location not found",

        (
            "The requested address could not "
            "be confirmed in the available map data. "
            "This does not mean it does not exist."
        )
    )


# ============================================================
# OFFLINE REGRESSION TESTS
# ============================================================

def run_offline_tests():

    for address in (
        "Rua 25 de Março, São Paulo, SP, Brasil",
        "Rua 7 de Setembro, Fortaleza, CE, Brasil"
    ):

        try:
            parse_address(address)

        except ValueError:
            pass

        else:
            raise AssertionError(
                "Missing house number accepted: "
                + address
            )

    parsed = parse_address(
        "Rua Tenente Jurandir Alencar, 66 - "
        "Messejana, Fortaleza - CE, 60840-000"
    )

    assert parsed["number"] == "66"
    assert norm(parsed["city"]) == "fortaleza"
    assert parsed["postal_code"] == "60840000"

    assert street_equal(
        "Rua 25 de Março",
        "Rua Vinte e Cinco de Março"
    )

    assert street_equal(
        "Avenida 9 de Julho",
        "Avenida Nove de Julho"
    )

    assert not street_equal(
        "Rua A",
        "Rua B"
    )

    assert postal_input(
        "60840-000"
    ) == "60840000"

    try:
        postal_input("123")

    except ValueError:
        pass

    else:
        raise AssertionError(
            "Invalid postal code accepted"
        )

    print(
        "LOC VIEW AI v5.9.3 — "
        "offline regression tests: PASS"
    )


run_offline_tests()


# ============================================================
# PROFESSIONAL INTERFACE — CSS
# ============================================================

CSS = """
.gradio-container {
    background: #f4f7fb !important;
    color: #0f172a !important;
    font-family:
        Inter,
        ui-sans-serif,
        system-ui,
        -apple-system,
        BlinkMacSystemFont,
        'Segoe UI',
        sans-serif !important;
}

/* Main white card */

#app-shell {
    max-width: 740px !important;
    margin: 34px auto 20px !important;
    padding: 34px !important;
    background: #ffffff !important;
    border: 1px solid #e2e8f0 !important;
    border-radius: 22px !important;
    box-shadow:
        0 14px 45px rgba(15,23,42,.045) !important;
}

/* Brand */

#brand .brandrow {
    display: flex;
    align-items: center;
    gap: 13px;
}

#brand .brandicon {
    display: grid;
    place-items: center;
    background: #eff6ff;
    border-radius: 13px;
    width: 46px;
    height: 46px;
    flex-shrink: 0;
    color: #2563eb;
}

#brand .brandname {
    font-size: 23px;
    line-height: 1.15;
    font-weight: 800;
    letter-spacing: -.8px;
    color: #0f172a;
}

#brand .brandname span {
    color: #2563eb;
}

#brand .tagline {
    font-size: 10px;
    letter-spacing: 1.1px;
    font-weight: 650;
    color: #64748b;
    margin-top: 5px;
}

/* Introduction */

#intro {
    border-top: 1px solid #e2e8f0;
    margin-top: 22px;
    padding-top: 25px;
    margin-bottom: 16px;
}

#intro h2 {
    font-size: 25px;
    letter-spacing: -.65px;
    color: #0f172a;
    margin: 0 0 8px;
    font-weight: 760;
}

#intro p {
    color: #64748b;
    font-size: 14px;
    line-height: 1.6;
    margin: 0;
}

/* ==========================================
   CLEAN INPUTS — NO GRAY OUTER BOX
   ========================================== */

/* Independent field labels */

.clean-label {
    font-size: 13px;
    font-weight: 650;
    color: #334155;
    margin: 0 0 7px 0;
    line-height: 1.4;
}

/* Remove the default textbox container */

#address-field,
#postal-field {
    background: transparent !important;
    background-color: transparent !important;
    border: none !important;
    box-shadow: none !important;
    padding: 0 !important;
    margin: 0 !important;
}

/* Individual text fields */

#address-field input,
#address-field textarea,
#postal-field input,
#postal-field textarea {
    width: 100% !important;
    background: #ffffff !important;
    background-color: #ffffff !important;
    border: 1px solid #cbd5e1 !important;
    border-radius: 10px !important;
    box-shadow: none !important;
    padding: 12px 14px !important;
    min-height: 45px !important;
    color: #0f172a !important;
    font-size: 14px !important;
}

/* Focus */

#address-field input:focus,
#address-field textarea:focus,
#postal-field input:focus,
#postal-field textarea:focus {
    border-color: #2563eb !important;
    outline: none !important;
    box-shadow:
        0 0 0 2px rgba(37,99,235,.10) !important;
}

/* Placeholder */

#address-field input::placeholder,
#address-field textarea::placeholder,
#postal-field input::placeholder,
#postal-field textarea::placeholder {
    color: #94a3b8 !important;
}

/* ==========================================
   PRIMARY BUTTON
   ========================================== */

#find-button {
    background: #2563eb !important;
    border: 1px solid #2563eb !important;
    border-radius: 10px !important;
    color: #ffffff !important;
    font-size: 13px !important;
    font-weight: 750 !important;
    min-height: 48px !important;
    height: 48px !important;
    line-height: 1.2 !important;
    padding: 12px 18px !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    text-align: center !important;
    margin-top: 12px !important;
}

#find-button:hover {
    background: #1d4ed8 !important;
}

/* ==========================================
   RESULT AREA
   ========================================== */

#result-container {
    border-top: 1px solid #e2e8f0;
    margin-top: 22px;
    padding-top: 24px;
}

.result {
    color: #0f172a;
}

.result .status {
    display: flex;
    align-items: center;
    gap: 9px;
    font-size: 16px;
    font-weight: 730;
    margin-bottom: 14px;
}

.result .status svg {
    width: 20px;
    height: 20px;
    flex-shrink: 0;
}

.result .status.ok svg {
    color: #16a34a;
}

.result .status.warn svg {
    color: #d97706;
}

.result .status.bad svg {
    color: #dc2626;
}

.result .address {
    font-size: 14px;
    line-height: 1.65;
    color: #334155;
    margin-bottom: 15px;
    overflow-wrap: anywhere;
}

.result .coords {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 11px;
    margin-bottom: 15px;
}

.result .coord {
    padding: 14px;
    background: #f8fafc;
    border-radius: 10px;
    min-width: 0;
}

.result .coord .k {
    font-size: 10px;
    letter-spacing: .6px;
    color: #64748b;
    font-weight: 700;
    margin-bottom: 6px;
}

.result .coord .v {
    font-size: 14px;
    color: #0f172a;
    font-weight: 650;
    overflow-wrap: anywhere;
}

.result .action {
    display: flex;
    justify-content: center;
    align-items: center;
    gap: 10px;
    text-decoration: none !important;
    border-radius: 10px;
    padding: 13px 12px;
    font-size: 13px;
    font-weight: 750;
    margin-top: 10px;
}

.result .action.primary {
    border: 1px solid #bfdbfe;
    color: #1d4ed8;
    background: #ffffff;
}

.result .action.primary:hover {
    background: #eff6ff;
}

.result .action.secondary {
    border: 1px solid #e2e8f0;
    color: #334155;
    background: #ffffff;
}

.result .action.secondary:hover {
    background: #f8fafc;
}

.result .action svg {
    width: 17px;
    height: 17px;
    flex-shrink: 0;
}

.result .detail {
    font-size: 13px;
    line-height: 1.6;
    color: #64748b;
    white-space: pre-line;
}

/* Footer */

#footer {
    text-align: center;
    color: #94a3b8;
    font-size: 11px;
    margin: 24px auto 8px;
    line-height: 1.7;
}

/* Mobile */

@media (max-width: 640px) {

    #app-shell {
        margin: 12px 8px !important;
        padding: 21px !important;
        border-radius: 16px !important;
    }

    #brand .brandname {
        font-size: 20px;
    }

    #brand .tagline {
        font-size: 8px;
        letter-spacing: .8px;
    }

    #intro h2 {
        font-size: 22px;
    }

    .result .coord .v {
        font-size: 12px;
    }

    .result .action {
        font-size: 11px;
        gap: 7px;
    }
}
"""


# ============================================================
# GRADIO APPLICATION
# ============================================================

with gr.Blocks(
    title="LOC VIEW AI",
    css=CSS,
    theme=gr.themes.Base(
        primary_hue="blue",
        neutral_hue="slate"
    )
) as app:

    with gr.Column(
        elem_id="app-shell"
    ):

        # ----------------------------------------
        # BRAND
        # ----------------------------------------

        gr.HTML(
            f"""
            <div id="brand">

                <div class="brandrow">

                    <div class="brandicon">
                        {PIN_ICON}
                    </div>

                    <div>

                        <div class="brandname">
                            LOC VIEW <span>AI</span>
                        </div>

                        <div class="tagline">
                            INTELLIGENT LOCATION VISUALIZATION
                        </div>

                    </div>

                </div>

            </div>
            """
        )

        # ----------------------------------------
        # INTRODUCTION
        # ----------------------------------------

        gr.HTML(
            """
            <div id="intro">

                <h2>
                    Find and explore a location.
                </h2>

                <p>
                    Enter an address to identify its
                    location and explore available
                    street-level imagery.
                </p>

            </div>
            """
        )

        # ----------------------------------------
        # CLEAN ADDRESS FIELD
        # ----------------------------------------

        gr.HTML(
            """
            <div class="clean-label">
                Street address
            </div>
            """
        )

        address = gr.Textbox(
            show_label=False,
            container=False,
            placeholder=(
                "Avenida Paulista, 1000, "
                "São Paulo, SP"
            ),
            lines=1,
            elem_id="address-field"
        )

        # ----------------------------------------
        # CLEAN POSTAL CODE FIELD
        # ----------------------------------------

        gr.HTML(
            """
            <div class="clean-label">
                Postal code (optional)
            </div>
            """
        )

        cep = gr.Textbox(
            show_label=False,
            container=False,
            placeholder="00000-000",
            lines=1,
            elem_id="postal-field"
        )

        # ----------------------------------------
        # SEARCH BUTTON
        # ----------------------------------------

        find_button = gr.Button(
            "FIND LOCATION ↗",
            variant="primary",
            elem_id="find-button"
        )

        # ----------------------------------------
        # RESULT
        # ----------------------------------------

        with gr.Column(
            visible=False,
            elem_id="result-container"
        ) as result_area:

            output = gr.HTML()

        # ----------------------------------------
        # SEARCH EVENTS
        # ----------------------------------------

        click_event = find_button.click(
            fn=search_location_ui,
            inputs=[address, cep],
            outputs=output
        )

        click_event.then(
            fn=lambda: gr.update(
                visible=True
            ),
            outputs=result_area
        )

        enter_event = address.submit(
            fn=search_location_ui,
            inputs=[address, cep],
            outputs=output
        )

        enter_event.then(
            fn=lambda: gr.update(
                visible=True
            ),
            outputs=result_area
        )

        # ----------------------------------------
        # FOOTER
        # ----------------------------------------

        gr.HTML(
            """
            <div id="footer">

                Location data © OpenStreetMap contributors.

                <br>

                LOC VIEW AI v5.9.3

            </div>
            """
        )


# ============================================================
# LAUNCH
# ============================================================

app.queue(
    default_concurrency_limit=1
)

app.launch(
    share=True
)


LOC VIEW AI v5.9.3 — offline regression tests: PASS


/tmp/ipykernel_1063/3255454251.py:1487: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3b06ddc26a38d1cef9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
